# 02 — Derivação de Chave Criptográfica via Argon2id

**Objetivo:** transformar o perfil STR (matéria-prima com ~10–22 bits de
entropia) em material criptográfico de comprimento fixo (32 bytes,
compatível com AES-256) através de uma função de derivação memory-hard.

**Pipeline desta etapa:**
1. Regeneração dos perfis sintéticos (mesma semente = mesmos dados)
2. Representação canônica do perfil (serialização determinística)
3. Derivação de chave com Argon2id (salt aleatório)
4. Benchmark de tempo no hardware local
5. Verificações: perfis distintos → chaves distintas; salts distintos → chaves distintas

**Contexto de segurança:** a KDF não *cria* entropia — ela torna a busca
exaustiva (força bruta) computacionalmente custosa, "esticando" o custo
por tentativa via consumo de memória e CPU.

In [1]:
import sys
from pathlib import Path

RAIZ = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(RAIZ))

import hashlib
import secrets
import time

import numpy as np
import pandas as pd

from argon2.low_level import hash_secret_raw, Type
from src.entropia import entropia_shannon

rng = np.random.default_rng(seed=2024)  # MESMA semente do Notebook 01

print(f"Raiz do projeto: {RAIZ}")
print("Bibliotecas criptográficas carregadas com sucesso.")

Raiz do projeto: /home/ricardo/Projetos/tcc-genetica-cripto
Bibliotecas criptográficas carregadas com sucesso.


## 1. Regeneração dos perfis (reprodutibilidade)

Como usamos a MESMA semente (`seed=2024`) do Notebook 01, os 5000 perfis
regenerados aqui são **idênticos aos da análise anterior**. Este é o
mecanismo que garante que qualquer pessoa rodando o pipeline obtenha
exatamente os mesmos resultados — requisito de reprodutibilidade do artigo.

In [2]:
PAINEL_STR = {
    "D3S1358":  {15: 0.21, 16: 0.26, 17: 0.22, 18: 0.22, 19: 0.09},
    "TH01":     {6: 0.23, 7: 0.19, 8: 0.31, 9: 0.17, 9.3: 0.10},
    "D21S11":   {28: 0.16, 29: 0.24, 30: 0.27, 31: 0.18, 32: 0.15},
    "D18S51":   {13: 0.15, 14: 0.21, 15: 0.33, 16: 0.19, 17: 0.12},
    "D5S818":   {11: 0.36, 12: 0.38, 13: 0.20, 14: 0.06},
}

N_INDIVIDUOS = 5000

def sortear_individuo(painel: dict, rng) -> dict:
    perfil = {}
    for locus, freqs in painel.items():
        alelos, probs = zip(*freqs.items())
        perfil[locus] = tuple(rng.choice(list(map(str, alelos)), size=2, p=probs))
    return perfil

perfis = [sortear_individuo(PAINEL_STR, rng) for _ in range(N_INDIVIDUOS)]

print(f"Perfis regenerados: {len(perfis)}")
print(f"Sanidade — indivíduo 0, D3S1358: {perfis[0]['D3S1358']}")



Perfis regenerados: 5000
Sanidade — indivíduo 0, D3S1358: (np.str_('17'), np.str_('16'))


## 2. Representação canônica do perfil

Antes de derivar a chave, o perfil genético precisa virar uma **sequência
de bytes determinística e sem ambiguidade**. Exigências:

1. **Ordem fixa de loci** — o mesmo dicionário em Python pode iterar em
   ordens diferentes em contextos distintos; fixamos a ordem alfabética
2. **Alelos ordenados** — o genótipo (16,17) é o mesmo que (17,16);
   sortear elimina essa ambiguidade
3. **Formato unívoco** — delimitadores que impedem "(1,15)" de ser lido
   como "(11,5)"

Analogia: é o "formato de registro" do mundo criptográfico — assim como
um arquivo DBF do Clipper tinha campos de tamanho fixo para nunca
ambiguar, aqui a canonização garante que bytes iguais ↔ perfil igual.

In [3]:
def perfil_canonico(perfil: dict) -> bytes:
    """
    Serializa o perfil STR em bytes canônicos.

    Regras: loci em ordem alfabética, alelos ordenados numericamente
    (quando decimais) e separador '|' entre campos.
    """
    partes = []
    for locus in sorted(perfil.keys()):
        a1, a2 = perfil[locus]
        # ordenação: tenta numérica (TH01 tem alelo 9.3), recorre à textual
        try:
            par_ordenado = sorted([a1, a2], key=float)
        except ValueError:
            par_ordenado = sorted([a1, a2])
        partes.append(f"{locus}:{par_ordenado[0]}:{par_ordenado[1]}")
    return "|".join(partes).encode("utf-8")


# Demonstração: heterozigoto em duas ordens → MESMOS bytes
p1 = {"TH01": ("9", "7"), "D21S11": ("30", "29")}
p2 = {"TH01": ("7", "9"), "D21S11": ("29", "30")}  # mesmos alelos, ordem trocada

c1, c2 = perfil_canonico(p1), perfil_canonico(p2)
print(f"Perfil 1 canonizado: {c1}")
print(f"Perfil 2 canonizado: {c2}")
print(f"São idênticos? {c1 == c2}  <-- TEM QUE SER True")

# E agora o indivíduo 0 do nosso dataset:
canon0 = perfil_canonico(perfis[0])
print(f"\nIndivíduo 0 ({len(canon0)} bytes): {canon0}")

Perfil 1 canonizado: b'D21S11:29:30|TH01:7:9'
Perfil 2 canonizado: b'D21S11:29:30|TH01:7:9'
São idênticos? True  <-- TEM QUE SER True

Indivíduo 0 (61 bytes): b'D18S51:13:14|D21S11:28:32|D3S1358:16:17|D5S818:11:11|TH01:7:9'


## 3. Derivação de chave com Argon2id

Agora aplicamos a KDF **Argon2id** (vencedora da Password Hashing
Competition, 2015). Propriedades:

- **Memory-hard**: consome um bloco de memória configurável por derivação,
  tornando ataques massivamente paralelos (GPU/ASIC) economicamente inviáveis
- **Salt aleatório**: garante que dois perfis iguais com salts diferentes
  produzam chaves sem relação entre si, e impede ataques de rainbow table
- **Saída de comprimento fixo**: 32 bytes, pronta para AES-256

**Modelo de segurança adotado:** perfil genético + salt → chave.
O salt precisa ser armazenado junto à chave (não é segredo); a chave
resultante NÃO permite recuperar o perfil (função de mão única).

In [7]:
# Distância de Hamming em nível de BIT (métrica correta para ~50%)
dist_bits_perfis = sum(bin(b1 ^ b2).count("1") for b1, b2 in zip(chave1, chave2))
print(f"Perfis distintos  -> {dist_bits_perfis}/256 bits diferentes (esperado ~128, tolerância ±16)")

dist_bits_salt = sum(bin(b1 ^ b2).count("1") for b1, b2 in zip(chave_salt1, chave_salt2))
print(f"Salts distintos   -> {dist_bits_salt}/256 bits diferentes (esperado ~128, tolerância ±16)")

Perfis distintos  -> 119/256 bits diferentes (esperado ~128, tolerância ±16)
Salts distintos   -> 130/256 bits diferentes (esperado ~128, tolerância ±16)


In [5]:
# Parâmetros do Argon2id (calibrados para hardware modesto)
PARAMS = dict(
    time_cost=3,      # iterações (passes sobre a memória)
    memory_cost=65536,  # 64 MiB de memória por derivação
    parallelism=1,    # threads (1 = compatível com qualquer máquina)
    hash_len=32,      # saída: 32 bytes (AES-256)
    type=Type.ID,     # variante híbrida (anti side-channel + anti GPU)
)

def derivar_chave(perfil: dict, salt: bytes, params: dict = PARAMS) -> bytes:
    """Deriva uma chave de 32 bytes do perfil STR canônico via Argon2id."""
    material = perfil_canonico(perfil)
    return hash_secret_raw(
        secret=material,
        salt=salt,
        **params,
    )

# Salt aleatório de 16 bytes (padrão mínimo recomendado)
salt_demo = secrets.token_bytes(16)

t0 = time.perf_counter()
chave_individuo0 = derivar_chave(perfis[0], salt_demo)
tempo_derivacao = time.perf_counter() - t0

print(f"Chave derivada (32 bytes): {chave_individuo0.hex()}")
print(f"Tempo de derivação: {tempo_derivacao:.3f} s")

Chave derivada (32 bytes): 6505b74ba534d88809bd4c596364e36d346a55e494b4b2c67c75ab68c8461900
Tempo de derivação: 0.303 s


In [6]:
# Teste 1: Benchmark em amostra dos 5000 perfis
N_BENCHMARK = 100
salt_fixo = secrets.token_bytes(16)

tempos = []
for i, perfil in enumerate(perfis[:N_BENCHMARK]):
    t0 = time.perf_counter()
    chave = derivar_chave(perfil, salt_fixo)
    tempos.append(time.perf_counter() - t0)

tempo_medio = sum(tempos) / len(tempos)
tempo_min = min(tempos)
tempo_max = max(tempos)

print(f"Benchmark ({N_BENCHMARK} perfis):")
print(f"  Médio: {tempo_medio:.3f} s")
print(f"  Mín: {tempo_min:.3f} s")
print(f"  Máx: {tempo_max:.3f} s")
print(f"  Desvio padrão: {(max(tempos) - min(tempos)):.3f} s")

# Teste 2: Perfis distintos → chaves distintas (diferenciação)
chave1 = derivar_chave(perfis[0], salt_fixo)
chave2 = derivar_chave(perfis[1], salt_fixo)
distancia_hamming = sum(b1 != b2 for b1, b2 in zip(chave1, chave2))
print(f"\nTeste de diferenciação:")
print(f"  Distância Hamming entre chaves de perfis adjacentes: {distancia_hamming}/{len(chave1)} bytes diferentes")
print(f"  Esperado (aleatório puro): ~{int(len(chave1) * 0.5)} bytes diferentes")

# Teste 3: Mesmo perfil + salt diferente → chaves sem correlação
salt_outro = secrets.token_bytes(16)
chave_salt1 = derivar_chave(perfis[0], salt_fixo)
chave_salt2 = derivar_chave(perfis[0], salt_outro)
distancia_salt = sum(b1 != b2 for b1, b2 in zip(chave_salt1, chave_salt2))
print(f"\nTeste de salting:")
print(f"  Mesma chave + salts diferentes: {distancia_salt}/{len(chave1)} bytes diferentes")
print(f"  Salts diferentes devem gerar chaves sem correlação (≈50%)")

Benchmark (100 perfis):
  Médio: 0.247 s
  Mín: 0.232 s
  Máx: 0.282 s
  Desvio padrão: 0.050 s

Teste de diferenciação:
  Distância Hamming entre chaves de perfis adjacentes: 32/32 bytes diferentes
  Esperado (aleatório puro): ~16 bytes diferentes

Teste de salting:
  Mesma chave + salts diferentes: 32/32 bytes diferentes
  Salts diferentes devem gerar chaves sem correlação (≈50%)


## 4. Leitura dos resultados

### Benchmark de desempenho
- Tempo médio: **__ ms** (no geladeiravelha, hardware modesto)
- Implicação de segurança: cada tentativa de força bruta custa __ ms
  + 64 MiB de memória, inviabilizando paralelização massiva

### Propriedades criptográficas verificadas
1. ✅ **Diferenciação**: perfis distintos geram chaves com ~50% de diferença (comportamento aleatório esperado)
2. ✅ **Independência de salt**: mesmo perfil com salt diferente gera chave sem correlação
3. ⚠️ **Entropia da fonte**: ~10–22 bits reais vs. 256 bits da saída — a KDF não cria entropia, apenas amplia o custo computacional do ataque

### Discussão metodológica
A abordagem adotada permite transformar perfil genético em chave utilizável,
mas **não elimina a necessidade de fator adicional** (senha, salt secreto)
quando o objetivo é segurança > 128 bits. Este é o gap que justifica futuras
extensões do trabalho (ver seção 5).